In [0]:
historical_df = spark.table(
    "finance.gold.finance_historical_patterns"
)

display(historical_df.limit(20))

In [0]:
historical_df.printSchema(); print(historical_df.columns)

root
 |-- fiscal_year: integer (nullable = true)
 |-- fiscal_month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- cost_center: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- department_name_en: string (nullable = true)
 |-- gl_account: string (nullable = true)
 |-- account_en: string (nullable = true)
 |-- gl_name_kr: string (nullable = true)
 |-- gl_name_en: string (nullable = true)
 |-- special_cost_check: string (nullable = true)
 |-- control_type: string (nullable = true)
 |-- actual_amount: decimal(18,2) (nullable = true)
 |-- actual_ytd: decimal(18,2) (nullable = true)
 |-- full_year_actual: decimal(18,2) (nullable = true)
 |-- historical_ytd_share: decimal(18,6) (nullable = true)

['fiscal_year', 'fiscal_month', 'month_name', 'cost_center', 'department_name', 'department_name_en', 'gl_account', 'account_en', 'gl_name_kr', 'gl_name_en', 'special_cost_check', 'control_type', 'actual_amount', 'actual_ytd', 'full_year_actual',

In [0]:
from pyspark.sql import functions as F


# 1. Historical Gold 데이터 불러오기
historical_df = spark.table(
    "finance.gold.finance_historical_patterns"
)


# 2. 현재 Reporting Cutoff Month 가져오기
reporting_period_df = spark.table(
    "finance.silver.reporting_period"
)

cutoff_month = (
    reporting_period_df
    .orderBy(F.desc("fiscal_year"))
    .select("cutoff_month")
    .first()["cutoff_month"]
)

print(f"Cutoff Month: {cutoff_month}")


# 3. G/L 단위 Actual을 Department 월 단위로 합산
department_monthly_df = (
    historical_df
    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name",
        "fiscal_month"
    )
    .agg(
        F.sum(
            F.col("actual_amount").cast("double")
        ).alias("monthly_actual")
    )
)


# 최근 3개월 범위 계산
recent_start_month = max(1, cutoff_month - 2)
recent_month_count = cutoff_month - recent_start_month + 1


# 4. Year × Department 단위 ML용 데이터 생성
ml_base_df = (
    department_monthly_df
    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )
    .agg(

        # 1월 ~ Cutoff Month 누적 Actual
        F.sum(
            F.when(
                F.col("fiscal_month") <= cutoff_month,
                F.col("monthly_actual")
            ).otherwise(0.0)
        ).alias("ytd_actual"),

        # Cutoff Month까지 월평균 Actual
        F.avg(
            F.when(
                F.col("fiscal_month") <= cutoff_month,
                F.col("monthly_actual")
            )
        ).alias("avg_monthly_actual"),

        # 최근 3개월 평균 Actual
        (
            F.sum(
                F.when(
                    (F.col("fiscal_month") >= recent_start_month)
                    & (F.col("fiscal_month") <= cutoff_month),
                    F.col("monthly_actual")
                ).otherwise(0.0)
            )
            / F.lit(recent_month_count)
        ).alias("recent_3m_avg_actual"),

        # Cutoff Month 한 달의 Actual
        F.max(
            F.when(
                F.col("fiscal_month") == cutoff_month,
                F.col("monthly_actual")
            )
        ).alias("cutoff_month_actual"),

        # 실제 1~12월 전체 Actual
        # ML이 맞혀야 할 Target
        F.sum(
            F.col("monthly_actual")
        ).alias("target_full_year_actual")
    )

    # 기존 Simple Annualization도 비교용으로 추가
    .withColumn(
        "simple_annualization",
        F.col("ytd_actual")
        / F.lit(cutoff_month)
        * F.lit(12.0)
    )
)


# 5. 데이터 확인
display(
    ml_base_df
    .orderBy("fiscal_year", "cost_center")
    .limit(30)
)


# 6. 행 수 검증
print(f"Total Rows: {ml_base_df.count()}")

display(
    ml_base_df
    .groupBy("fiscal_year")
    .count()
    .orderBy("fiscal_year")
)

Cutoff Month: 8


fiscal_year,cost_center,department_name,ytd_actual,avg_monthly_actual,recent_3m_avg_actual,cutoff_month_actual,target_full_year_actual,simple_annualization
2021,410100,경영기획팀,4.72898135E8,5.9112266875E7,5.9070334333333336E7,5.7634779E7,7.16708515E8,7.093472025E8
2021,410110,재무회계팀,3.9709971E8,4.963746375E7,4.9973942666666664E7,4.9287275E7,6.01547168E8,5.95649565E8
2021,410120,정보기술팀,2.95011473E8,3.6876434125E7,3.7119349333333336E7,3.6971778E7,4.42388133E8,4.425172095E8
2021,410130,인사팀,3.1332962E8,3.91662025E7,3.9408783666666664E7,3.8928582E7,4.71986506E8,4.6999443E8
2021,410140,구매팀,4.89710743E8,6.1213842875E7,6.1705802E7,6.1692664E7,7.36086513E8,7.345661145E8
2021,410150,국내영업팀,4.41832646E8,5.522908075E7,5.4866425333333336E7,5.3363071E7,6.68192334E8,6.62748969E8
2021,410160,해외영업팀,5.41725658E8,6.771570725E7,6.761798133333333E7,6.5810156E7,8.22268977E8,8.12588487E8
2021,410170,연구개발팀,6.71472035E8,8.3934004375E7,8.434656733333333E7,8.2700685E7,1.01137068E9,1.0072080525E9
2021,410180,생산관리팀,6.11610208E8,7.6451276E7,7.6295529E7,7.4405968E7,9.23347005E8,9.17415312E8
2021,410190,품질관리팀,4.83240984E8,6.0405123E7,6.0889807E7,6.0267997E7,7.27970834E8,7.24861476E8


Total Rows: 220


fiscal_year,count
2021,44
2022,44
2023,44
2024,44
2025,44


In [0]:
# 현재 Reporting Year 확인
reporting_year = (
    reporting_period_df
    .orderBy(F.desc("fiscal_year"))
    .select("fiscal_year")
    .first()["fiscal_year"]
)

# Reporting Year 바로 전 연도를 Backtest 연도로 사용
test_year = reporting_year - 1


# Train: Test Year보다 이전 연도
train_df = (
    ml_base_df
    .filter(F.col("fiscal_year") < test_year)
)

# Test: 2025
test_df = (
    ml_base_df
    .filter(F.col("fiscal_year") == test_year)
)


print(f"Reporting Year : {reporting_year}")
print(f"Test Year      : {test_year}")

print(f"Train Rows     : {train_df.count()}")
print(f"Test Rows      : {test_df.count()}")


display(
    train_df
    .groupBy("fiscal_year")
    .count()
    .orderBy("fiscal_year")
)

Reporting Year : 2026
Test Year      : 2025
Train Rows     : 176
Test Rows      : 44


fiscal_year,count
2021,44
2022,44
2023,44
2024,44


In [0]:
# ML에서 사용할 Feature 컬럼
feature_columns = [
    "ytd_actual",
    "recent_3m_avg_actual",
    "cutoff_month_actual"
]

# ML이 예측해야 할 Target 컬럼
target_column = "target_full_year_actual"


# Train Feature
X_train_df = train_df.select(
    "cost_center",
    "department_name",
    *feature_columns
)

# Train Target
y_train_df = train_df.select(
    "cost_center",
    "department_name",
    target_column
)


# Test Feature
X_test_df = test_df.select(
    "cost_center",
    "department_name",
    *feature_columns
)

# Test Target
y_test_df = test_df.select(
    "cost_center",
    "department_name",
    target_column
)


print("Feature Columns:")
print(feature_columns)

print(f"\nTrain Feature Rows : {X_train_df.count()}")
print(f"Train Target Rows  : {y_train_df.count()}")
print(f"Test Feature Rows  : {X_test_df.count()}")
print(f"Test Target Rows   : {y_test_df.count()}")


display(X_train_df.limit(10))

Feature Columns:
['ytd_actual', 'recent_3m_avg_actual', 'cutoff_month_actual']

Train Feature Rows : 176
Train Target Rows  : 176
Test Feature Rows  : 44
Test Target Rows   : 44


cost_center,department_name,ytd_actual,recent_3m_avg_actual,cutoff_month_actual
410100,경영기획팀,4.72898135E8,5.9070334333333336E7,5.7634779E7
410130,인사팀,3.1332962E8,3.9408783666666664E7,3.8928582E7
410140,구매팀,4.89710743E8,6.1705802E7,6.1692664E7
410150,국내영업팀,4.41832646E8,5.4866425333333336E7,5.3363071E7
410160,해외영업팀,5.41725658E8,6.761798133333333E7,6.5810156E7
410170,연구개발팀,6.71472035E8,8.434656733333333E7,8.2700685E7
410180,생산관리팀,6.11610208E8,7.6295529E7,7.4405968E7
410190,품질관리팀,4.83240984E8,6.0889807E7,6.0267997E7
410210,고객지원팀,3.58251299E8,4.4747580333333336E7,4.3616129E7
410220,마케팅팀,4.61239513E8,5.7503610333333336E7,5.5700318E7


In [0]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# --------------------------------------------------
# 1. Spark DataFrame → Pandas DataFrame 변환
# --------------------------------------------------

train_pd = (
    train_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        *feature_columns,
        target_column
    )
    .orderBy("fiscal_year", "cost_center")
    .toPandas()
)

test_pd = (
    test_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        *feature_columns,
        target_column,
        "simple_annualization"
    )
    .orderBy("fiscal_year", "cost_center")
    .toPandas()
)


# --------------------------------------------------
# 2. Feature(X) / Target(y) 분리
# --------------------------------------------------

X_train = train_pd[feature_columns]
y_train = train_pd[target_column]

X_test = test_pd[feature_columns]
y_test = test_pd[target_column]


# --------------------------------------------------
# 3. Linear Regression 모델 생성 및 학습
# --------------------------------------------------

model = LinearRegression()

model.fit(
    X_train,
    y_train
)


# --------------------------------------------------
# 4. 2025년 데이터 예측
# --------------------------------------------------

ml_prediction = model.predict(X_test)


# --------------------------------------------------
# 5. 예측 결과를 Test 데이터에 추가
# --------------------------------------------------

result_pd = test_pd.copy()

result_pd["ml_prediction"] = ml_prediction

result_pd["ml_error_amount"] = (
    result_pd["ml_prediction"]
    - result_pd[target_column]
)

result_pd["ml_ape"] = (
    np.abs(
        result_pd["ml_prediction"]
        - result_pd[target_column]
    )
    / np.abs(result_pd[target_column])
)


# --------------------------------------------------
# 6. 모델 성능 계산
# --------------------------------------------------

mae = mean_absolute_error(
    y_test,
    ml_prediction
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        ml_prediction
    )
)

valid_mask = y_test != 0

mape = np.mean(
    np.abs(
        (
            y_test[valid_mask]
            - ml_prediction[valid_mask]
        )
        / y_test[valid_mask]
    )
) * 100


print("=== Linear Regression Result ===")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : {mape:.4f}%")

print("\nModel Coefficients:")

for feature, coefficient in zip(
    feature_columns,
    model.coef_
):
    print(
        f"{feature:25s}: {coefficient:.6f}"
    )

print(
    f"Intercept                : "
    f"{model.intercept_:,.2f}"
)


# --------------------------------------------------
# 7. 2025 부서별 예측 결과 확인
# --------------------------------------------------

display(
    result_pd[
        [
            "cost_center",
            "department_name",
            target_column,
            "simple_annualization",
            "ml_prediction",
            "ml_error_amount",
            "ml_ape"
        ]
    ]
)

=== Linear Regression Result ===
MAE  : 6,316,202.52
RMSE : 15,339,662.63
MAPE : 0.7883%

Model Coefficients:
ytd_actual               : 1.778935
recent_3m_avg_actual     : -0.638169
cutoff_month_actual      : -1.662310
Intercept                : 9,593,556.45


cost_center,department_name,target_full_year_actual,simple_annualization,ml_prediction,ml_error_amount,ml_ape
410100,경영기획팀,8.90016047E8,8.8077366E8,8.893832696508335E8,-632777.3491665125,7.109729664981113E-4
410110,재무회계팀,7.43675445E8,7.372597905E8,7.443576071714557E8,682162.1714557409,9.172847860476836E-4
410120,정보기술팀,5.49216339E8,5.490007605E8,5.553875072182863E8,6171168.218286276,0.011236315783180434
410130,인사팀,5.83039604E8,5.79484362E8,5.86326722303888E8,3287118.3038879633,0.005637898834549777
410140,구매팀,9.10496769E8,9.088056375E8,9.124829628679712E8,1986193.8679711819,0.0021814397761703444
410150,국내영업팀,8.29696174E8,8.21724966E8,8.289174351836344E8,-778738.8163655996,9.385831112264471E-4
410160,해외영업팀,1.018208935E9,1.0066617465E9,1.0145398305856365E9,-3669104.4143635035,0.0036034887224433002
410170,연구개발팀,1.252445823E9,1.2482187075E9,1.2524857520913355E9,39929.09133553505,3.188089305123983E-5
410180,생산관리팀,1.143489082E9,1.1379872025E9,1.144555680333788E9,1066598.333787918,9.327577766832741E-4
410190,품질관리팀,9.02979208E8,8.99842188E8,9.050054985833383E8,2026290.5833382607,0.0022440058036621597


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# --------------------------------------------------
# 1. Historical Pattern Gold 다시 읽기
# --------------------------------------------------

historical_df = spark.table(
    "finance.gold.finance_historical_patterns"
)

reporting_period_df = spark.table(
    "finance.silver.reporting_period"
)

cutoff_month = (
    reporting_period_df
    .orderBy(F.desc("fiscal_year"))
    .select("cutoff_month")
    .first()["cutoff_month"]
)

print(f"Cutoff Month: {cutoff_month}")


# --------------------------------------------------
# 2. Year × Department별 YTD Share 계산
# --------------------------------------------------

department_pattern_df = (
    historical_df

    # 현재 Reporting Cutoff와 같은 월만 사용
    .filter(
        F.col("fiscal_month") == cutoff_month
    )

    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )

    .agg(
        F.sum(
            F.col("actual_ytd").cast("double")
        ).alias("department_ytd_actual"),

        F.sum(
            F.col("full_year_actual").cast("double")
        ).alias("department_full_year_actual")
    )

    .withColumn(
        "department_ytd_share",

        F.when(
            F.col("department_full_year_actual") > 0,

            F.col("department_ytd_actual")
            / F.col("department_full_year_actual")
        )
    )
)


# --------------------------------------------------
# 3. 해당 연도보다 이전 연도의 평균 YTD Share 계산
# --------------------------------------------------

prior_year_window = (
    Window
    .partitionBy(
        "cost_center",
        "department_name"
    )
    .orderBy("fiscal_year")
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)


seasonal_pattern_df = (
    department_pattern_df

    .withColumn(
        "historical_avg_ytd_share",

        F.avg(
            "department_ytd_share"
        ).over(prior_year_window)
    )

    .withColumn(
        "history_year_count",

        F.count(
            "department_ytd_share"
        ).over(prior_year_window)
    )
)


# --------------------------------------------------
# 4. 기존 ML Base 데이터와 결합
# --------------------------------------------------

ml_seasonal_df = (
    ml_base_df

    .join(
        seasonal_pattern_df.select(
            "fiscal_year",
            "cost_center",
            "department_name",
            "historical_avg_ytd_share",
            "history_year_count"
        ),

        on=[
            "fiscal_year",
            "cost_center",
            "department_name"
        ],

        how="left"
    )

    # Historical Seasonality를 이용한 연말 예상값
    .withColumn(
        "seasonal_eac_feature",

        F.when(
            F.col("historical_avg_ytd_share") > 0,

            F.col("ytd_actual")
            / F.col("historical_avg_ytd_share")
        )
    )
)


# --------------------------------------------------
# 5. 주요 Scenario 검증
# --------------------------------------------------

display(
    ml_seasonal_df
    .filter(
        F.col("cost_center").isin(
            "410500",   # 마케팅팀 3
            "410260"    # 정보기술팀 2
        )
    )
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "historical_avg_ytd_share",
        "history_year_count",
        "seasonal_eac_feature",
        "target_full_year_actual"
    )
    .orderBy(
        "cost_center",
        "fiscal_year"
    )
)

Cutoff Month: 8


fiscal_year,cost_center,department_name,ytd_actual,historical_avg_ytd_share,history_year_count,seasonal_eac_feature,target_full_year_actual
2021,410260,정보기술팀 2,4.61424079E8,null,0,null,7.08739421E8
2022,410260,정보기술팀 2,5.3742816E8,0.6510489826415342,1,8.254803775584831E8,8.25056206E8
2023,410260,정보기술팀 2,6.19622766E8,0.6512163387923078,2,9.514852885127258E8,9.51783847E8
2024,410260,정보기술팀 2,6.85694594E8,0.6511482469436488,3,1.0530545036687181E9,1.069627319E9
2025,410260,정보기술팀 2,8.73620945E8,0.6486260227403688,4,1.3468792715239115E9,1.317164518E9
2021,410500,마케팅팀 3,3.72119845E8,null,0,null,6.53062221E8
2022,410500,마케팅팀 3,3.91438744E8,0.569807643183206,1,6.86966467864917E8,6.96884583E8
2023,410500,마케팅팀 3,4.04353264E8,0.5657528700200427,2,7.147171237251967E8,7.12781494E8
2024,410500,마케팅팀 3,4.25570148E8,0.5662649895973813,3,7.515388657571496E8,7.53234476E8
2025,410500,마케팅팀 3,4.51717569E8,0.5659463090446708,4,7.981632917838243E8,7.86714864E8


In [0]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# --------------------------------------------------
# 1. Test Year 확인
# --------------------------------------------------

reporting_year = (
    reporting_period_df
    .orderBy(F.desc("fiscal_year"))
    .select("fiscal_year")
    .first()["fiscal_year"]
)

test_year = reporting_year - 1


# --------------------------------------------------
# 2. Historical Feature가 존재하는 데이터만 사용
# --------------------------------------------------

train_seasonal_df = (
    ml_seasonal_df
    .filter(
        (F.col("fiscal_year") < test_year)
        &
        F.col("seasonal_eac_feature").isNotNull()
    )
)

test_seasonal_df = (
    ml_seasonal_df
    .filter(
        F.col("fiscal_year") == test_year
    )
)


print(
    f"Train Rows : {train_seasonal_df.count()}"
)

print(
    f"Test Rows  : {test_seasonal_df.count()}"
)


# --------------------------------------------------
# 3. Spark → Pandas
# --------------------------------------------------

train_pd = (
    train_seasonal_df
    .orderBy(
        "fiscal_year",
        "cost_center"
    )
    .toPandas()
)

test_pd = (
    test_seasonal_df
    .orderBy(
        "cost_center"
    )
    .toPandas()
)


# --------------------------------------------------
# 4. Feature 정의
# --------------------------------------------------

base_features = [
    "ytd_actual",
    "recent_3m_avg_actual",
    "cutoff_month_actual"
]

seasonal_features = [
    "ytd_actual",
    "recent_3m_avg_actual",
    "cutoff_month_actual",
    "seasonal_eac_feature"
]

target = "target_full_year_actual"


# --------------------------------------------------
# 5. 기존 Linear Regression
#    동일한 2022~2024 데이터로 다시 학습
# --------------------------------------------------

base_model = LinearRegression()

base_model.fit(
    train_pd[base_features],
    train_pd[target]
)

base_prediction = base_model.predict(
    test_pd[base_features]
)


# --------------------------------------------------
# 6. Seasonal Feature 추가 Linear Regression
# --------------------------------------------------

seasonal_model = LinearRegression()

seasonal_model.fit(
    train_pd[seasonal_features],
    train_pd[target]
)

seasonal_prediction = seasonal_model.predict(
    test_pd[seasonal_features]
)


# --------------------------------------------------
# 7. 평가 함수
# --------------------------------------------------

def evaluate_model(actual, prediction):

    mae = mean_absolute_error(
        actual,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )

    mask = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[mask]
                - prediction[mask]
            )
            / actual[mask]
        )
    ) * 100

    return mae, rmse, mape


# --------------------------------------------------
# 8. 성능 계산
# --------------------------------------------------

base_mae, base_rmse, base_mape = (
    evaluate_model(
        test_pd[target],
        base_prediction
    )
)

seasonal_mae, seasonal_rmse, seasonal_mape = (
    evaluate_model(
        test_pd[target],
        seasonal_prediction
    )
)


print("=== Same Training Period Comparison ===")

print("\n[Base Linear Regression]")
print(f"MAE  : {base_mae:,.2f}")
print(f"RMSE : {base_rmse:,.2f}")
print(f"MAPE : {base_mape:.4f}%")

print("\n[Linear Regression + Historical Seasonality]")
print(f"MAE  : {seasonal_mae:,.2f}")
print(f"RMSE : {seasonal_rmse:,.2f}")
print(f"MAPE : {seasonal_mape:.4f}%")

print("\n[Existing Seasonal EAC Baseline]")
print("MAPE : 0.3213%")


# --------------------------------------------------
# 9. 부서별 결과
# --------------------------------------------------

comparison_pd = test_pd[
    [
        "cost_center",
        "department_name",
        target,
        "seasonal_eac_feature"
    ]
].copy()

comparison_pd["base_ml_prediction"] = (
    base_prediction
)

comparison_pd["seasonal_ml_prediction"] = (
    seasonal_prediction
)

comparison_pd["base_ml_ape"] = (
    np.abs(
        comparison_pd["base_ml_prediction"]
        - comparison_pd[target]
    )
    / np.abs(comparison_pd[target])
)

comparison_pd["seasonal_ml_ape"] = (
    np.abs(
        comparison_pd["seasonal_ml_prediction"]
        - comparison_pd[target]
    )
    / np.abs(comparison_pd[target])
)


display(
    comparison_pd.sort_values(
        "seasonal_ml_ape",
        ascending=False
    )
)

Train Rows : 132
Test Rows  : 44
=== Same Training Period Comparison ===

[Base Linear Regression]
MAE  : 5,826,146.79
RMSE : 14,531,748.16
MAPE : 0.7568%

[Linear Regression + Historical Seasonality]
MAE  : 9,899,964.50
RMSE : 37,665,827.79
MAPE : 1.1030%

[Existing Seasonal EAC Baseline]
MAPE : 0.3213%


cost_center,department_name,target_full_year_actual,seasonal_eac_feature,base_ml_prediction,seasonal_ml_prediction,base_ml_ape,seasonal_ml_ape
410450,연구개발팀 3,8.67956264E8,9.337579890261233E8,8.492506237829986E8,1.113761881922072E9,0.02155136265829339,0.2832004654120129
410320,생산관리팀 2,1.481639304E9,1.4833807487564223E9,1.4671021477845078E9,1.5134293853655539E9,0.009811535220647906,0.02145601920773145
410380,경영기획팀 3,2.98950786E8,2.9931712049809223E8,3.081063654690068E8,3.044167164863461E8,0.030625707968557663,0.018283713381326002
410260,정보기술팀 2,1.317164518E9,1.3468792715239115E9,1.3276508072967293E9,1.3042121907795496E9,0.007961260080594827,0.009833492356837387
410310,연구개발팀 2,1.142237765E9,1.1422458487387862E9,1.134720357728079E9,1.131346291703259E9,0.006581298134474585,0.009535206793605715
410460,생산관리팀 3,1.031548807E9,1.0312770300549372E9,1.0331403545408224E9,1.0232298249264382E9,0.0015428717769070007,0.008064554984805279
410170,연구개발팀,1.252445823E9,1.2518174146408062E9,1.251311313652431E9,1.2434580900400686E9,9.058350682598658E-4,0.007176145103349012
410250,재무회계팀 2,9.62334041E8,9.618768918682346E8,9.607973653649144E8,9.555358518926494E8,0.0015968214462088043,0.00706427167461136
410180,생산관리팀,1.143489082E9,1.1437610511643977E9,1.1433775679918404E9,1.135487039325635E9,9.752083331184569E-5,0.006997917864129676
410100,경영기획팀,8.90016047E8,8.897737609694924E8,8.886876871271961E8,8.842262310952711E8,0.0014925122724263943,0.006505293836268201


In [0]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# --------------------------------------------------
# 1. Residual Model용 Train / Test 데이터 준비
# --------------------------------------------------

train_residual_pd = (
    train_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "target_full_year_actual"
    )
    .orderBy("fiscal_year", "cost_center")
    .toPandas()
)

test_residual_pd = (
    test_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "target_full_year_actual"
    )
    .orderBy("cost_center")
    .toPandas()
)


# --------------------------------------------------
# 2. 현재까지 월평균 Actual 계산
# --------------------------------------------------

train_residual_pd["ytd_monthly_avg"] = (
    train_residual_pd["ytd_actual"]
    / cutoff_month
)

test_residual_pd["ytd_monthly_avg"] = (
    test_residual_pd["ytd_actual"]
    / cutoff_month
)


# --------------------------------------------------
# 3. 최근 지출 속도를 비율로 Feature화
# --------------------------------------------------

train_residual_pd["recent_3m_momentum"] = (
    train_residual_pd["recent_3m_avg_actual"]
    / train_residual_pd["ytd_monthly_avg"]
    - 1
)

test_residual_pd["recent_3m_momentum"] = (
    test_residual_pd["recent_3m_avg_actual"]
    / test_residual_pd["ytd_monthly_avg"]
    - 1
)


train_residual_pd["cutoff_month_momentum"] = (
    train_residual_pd["cutoff_month_actual"]
    / train_residual_pd["ytd_monthly_avg"]
    - 1
)

test_residual_pd["cutoff_month_momentum"] = (
    test_residual_pd["cutoff_month_actual"]
    / test_residual_pd["ytd_monthly_avg"]
    - 1
)


# --------------------------------------------------
# 4. ML이 학습할 정답: Seasonal EAC의 오차율
# --------------------------------------------------

train_residual_pd["residual_rate"] = (
    (
        train_residual_pd["target_full_year_actual"]
        - train_residual_pd["seasonal_eac_feature"]
    )
    / train_residual_pd["seasonal_eac_feature"]
)


# --------------------------------------------------
# 5. Residual Model에서 사용할 Feature
# --------------------------------------------------

residual_features = [
    "recent_3m_momentum",
    "cutoff_month_momentum",
    "historical_avg_ytd_share"
]


# --------------------------------------------------
# 6. Linear Regression으로 Residual Rate 학습
# --------------------------------------------------

residual_model = LinearRegression()

residual_model.fit(
    train_residual_pd[residual_features],
    train_residual_pd["residual_rate"]
)


# --------------------------------------------------
# 7. 2025 Residual Rate 예측
# --------------------------------------------------

predicted_residual_rate = residual_model.predict(
    test_residual_pd[residual_features]
)


# --------------------------------------------------
# 8. Seasonal EAC + ML 보정
# --------------------------------------------------

test_residual_pd["predicted_residual_rate"] = (
    predicted_residual_rate
)

test_residual_pd["residual_ml_prediction"] = (
    test_residual_pd["seasonal_eac_feature"]
    * (
        1
        + test_residual_pd["predicted_residual_rate"]
    )
)


# --------------------------------------------------
# 9. 평가 함수
# --------------------------------------------------

def evaluate(actual, prediction):

    mae = mean_absolute_error(
        actual,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )

    mask = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[mask]
                - prediction[mask]
            )
            / actual[mask]
        )
    ) * 100

    return mae, rmse, mape


actual = test_residual_pd[
    "target_full_year_actual"
]


# Seasonal EAC 단독 성능
seasonal_mae, seasonal_rmse, seasonal_mape = evaluate(
    actual,
    test_residual_pd["seasonal_eac_feature"]
)


# Residual ML 보정 후 성능
residual_mae, residual_rmse, residual_mape = evaluate(
    actual,
    test_residual_pd["residual_ml_prediction"]
)


# --------------------------------------------------
# 10. 결과 출력
# --------------------------------------------------

print("=== Seasonal EAC ===")
print(f"MAE  : {seasonal_mae:,.2f}")
print(f"RMSE : {seasonal_rmse:,.2f}")
print(f"MAPE : {seasonal_mape:.4f}%")


print("\n=== Seasonal EAC + Residual ML ===")
print(f"MAE  : {residual_mae:,.2f}")
print(f"RMSE : {residual_rmse:,.2f}")
print(f"MAPE : {residual_mape:.4f}%")


print("\n=== Residual Model Coefficients ===")

for feature, coefficient in zip(
    residual_features,
    residual_model.coef_
):
    print(
        f"{feature:30s}: {coefficient:.6f}"
    )

print(
    f"Intercept                     : "
    f"{residual_model.intercept_:.6f}"
)


# --------------------------------------------------
# 11. 부서별 오차 계산
# --------------------------------------------------

test_residual_pd["seasonal_ape"] = (
    np.abs(
        test_residual_pd["seasonal_eac_feature"]
        - actual
    )
    / np.abs(actual)
)

test_residual_pd["residual_ml_ape"] = (
    np.abs(
        test_residual_pd["residual_ml_prediction"]
        - actual
    )
    / np.abs(actual)
)


display(
    test_residual_pd[
        [
            "cost_center",
            "department_name",
            "target_full_year_actual",
            "seasonal_eac_feature",
            "predicted_residual_rate",
            "residual_ml_prediction",
            "seasonal_ape",
            "residual_ml_ape"
        ]
    ]
    .sort_values(
        "residual_ml_ape",
        ascending=False
    )
)

=== Seasonal EAC ===
MAE  : 2,952,601.19
RMSE : 11,039,994.17
MAPE : 0.3213%

=== Seasonal EAC + Residual ML ===
MAE  : 8,260,893.96
RMSE : 36,575,940.16
MAPE : 0.9073%

=== Residual Model Coefficients ===
recent_3m_momentum            : -0.098397
cutoff_month_momentum         : 0.249472
historical_avg_ytd_share      : -0.385191
Intercept                     : 0.257904


cost_center,department_name,target_full_year_actual,seasonal_eac_feature,predicted_residual_rate,residual_ml_prediction,seasonal_ape,residual_ml_ape
410450,연구개발팀 3,8.67956264E8,9.337579890261233E8,0.18414386052208292,1.1057037899187303E9,0.07581225892981548,0.27391648148613423
410320,생산관리팀 2,1.481639304E9,1.4833807487564223E9,0.029656904115519633,1.5273732293890994E9,0.0011753500003144363,0.030867111358095667
410500,마케팅팀 3,7.86714864E8,7.981632917838243E8,-0.006576325160314089,7.929143104460272E8,0.014552194584980315,0.007880169461279124
410340,물류팀 2,7.26068591E8,7.253724644909654E8,-0.004213490774630291,7.223161143036618E8,9.58761359000356E-4,0.005168212401489457
410460,생산관리팀 3,1.031548807E9,1.0312770300549372E9,-0.004545704331705147,1.0265891495922284E9,2.6346494050354373E-4,0.004807971638487472
410310,연구개발팀 2,1.142237765E9,1.1422458487387862E9,-0.004544480634421033,1.137054934599445E9,7.07710691584475E-6,0.004537435689280415
410220,마케팅팀,8.59890083E8,8.589520766577805E8,-0.00336219067434107,8.560641159959358E8,0.0010908444704315445,0.004449367517667026
410330,품질관리팀 2,7.00939463E8,7.017342048239614E8,-0.005532088735250518,6.978521489343147E8,0.0011338237692594818,0.0044045373796927625
410100,경영기획팀,8.90016047E8,8.897737609694924E8,-0.003963400428259767,8.862472312642117E8,2.7222658661520127E-4,0.004234548071904984
410250,재무회계팀 2,9.62334041E8,9.618768918682346E8,-0.003691272669171519,9.583263419861737E8,4.750420459930146E-4,0.004164561205443475


In [0]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# --------------------------------------------------
# 1. Gradient Boosting용 Train / Test 데이터 준비
# --------------------------------------------------

train_gb_pd = (
    train_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "simple_annualization",
        "target_full_year_actual"
    )
    .orderBy(
        "fiscal_year",
        "cost_center"
    )
    .toPandas()
)


test_gb_pd = (
    test_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "simple_annualization",
        "target_full_year_actual"
    )
    .orderBy("cost_center")
    .toPandas()
)


# --------------------------------------------------
# 2. YTD 월평균 계산
# --------------------------------------------------

train_gb_pd["ytd_monthly_avg"] = (
    train_gb_pd["ytd_actual"]
    / cutoff_month
)

test_gb_pd["ytd_monthly_avg"] = (
    test_gb_pd["ytd_actual"]
    / cutoff_month
)


# --------------------------------------------------
# 3. 최근 3개월 Momentum 계산
# --------------------------------------------------

train_gb_pd["recent_3m_momentum"] = (
    train_gb_pd["recent_3m_avg_actual"]
    / train_gb_pd["ytd_monthly_avg"]
    - 1
)

test_gb_pd["recent_3m_momentum"] = (
    test_gb_pd["recent_3m_avg_actual"]
    / test_gb_pd["ytd_monthly_avg"]
    - 1
)


# --------------------------------------------------
# 4. Cutoff Month Momentum 계산
# --------------------------------------------------

train_gb_pd["cutoff_month_momentum"] = (
    train_gb_pd["cutoff_month_actual"]
    / train_gb_pd["ytd_monthly_avg"]
    - 1
)

test_gb_pd["cutoff_month_momentum"] = (
    test_gb_pd["cutoff_month_actual"]
    / test_gb_pd["ytd_monthly_avg"]
    - 1
)


# --------------------------------------------------
# 5. Feature / Target 정의
# --------------------------------------------------

gb_features = [
    "ytd_actual",
    "recent_3m_momentum",
    "cutoff_month_momentum",
    "historical_avg_ytd_share"
]

target = "target_full_year_actual"


X_train_gb = train_gb_pd[gb_features]
y_train_gb = train_gb_pd[target]

X_test_gb = test_gb_pd[gb_features]
y_test_gb = test_gb_pd[target]


# --------------------------------------------------
# 6. Gradient Boosting 모델 생성
# --------------------------------------------------

gb_model = GradientBoostingRegressor(

    # 작은 Tree를 몇 개 만들 것인지
    n_estimators=100,

    # 각 Tree의 보정값을 얼마나 반영할지
    learning_rate=0.05,

    # Tree 하나가 너무 복잡해지지 않도록 제한
    max_depth=2,

    # 한 Leaf에 너무 적은 데이터만 들어가지 않도록 제한
    min_samples_leaf=5,

    # 실행할 때마다 같은 결과가 나오도록 고정
    random_state=42
)


# --------------------------------------------------
# 7. 모델 학습
# --------------------------------------------------

gb_model.fit(
    X_train_gb,
    y_train_gb
)


# --------------------------------------------------
# 8. 2025 예측
# --------------------------------------------------

gb_prediction = gb_model.predict(
    X_test_gb
)


# --------------------------------------------------
# 9. 평가 함수
# --------------------------------------------------

def evaluate_model(actual, prediction):

    mae = mean_absolute_error(
        actual,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )

    mask = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[mask]
                - prediction[mask]
            )
            / actual[mask]
        )
    ) * 100

    return mae, rmse, mape


# --------------------------------------------------
# 10. Gradient Boosting 성능
# --------------------------------------------------

gb_mae, gb_rmse, gb_mape = evaluate_model(
    y_test_gb,
    gb_prediction
)


# Seasonal EAC 성능
seasonal_mae, seasonal_rmse, seasonal_mape = evaluate_model(
    y_test_gb,
    test_gb_pd["seasonal_eac_feature"]
)


# Simple Annualization 성능
simple_mae, simple_rmse, simple_mape = evaluate_model(
    y_test_gb,
    test_gb_pd["simple_annualization"]
)


# --------------------------------------------------
# 11. 전체 성능 비교
# --------------------------------------------------

print("=== 2025 Forecast Comparison ===")

print("\n[Simple Annualization]")
print(f"MAE  : {simple_mae:,.2f}")
print(f"RMSE : {simple_rmse:,.2f}")
print(f"MAPE : {simple_mape:.4f}%")

print("\n[Seasonal EAC]")
print(f"MAE  : {seasonal_mae:,.2f}")
print(f"RMSE : {seasonal_rmse:,.2f}")
print(f"MAPE : {seasonal_mape:.4f}%")

print("\n[Gradient Boosting]")
print(f"MAE  : {gb_mae:,.2f}")
print(f"RMSE : {gb_rmse:,.2f}")
print(f"MAPE : {gb_mape:.4f}%")


# --------------------------------------------------
# 12. Feature Importance 확인
# --------------------------------------------------

print("\n=== Feature Importance ===")

for feature, importance in sorted(
    zip(
        gb_features,
        gb_model.feature_importances_
    ),
    key=lambda x: x[1],
    reverse=True
):
    print(
        f"{feature:30s}: {importance:.4f}"
    )


# --------------------------------------------------
# 13. 부서별 결과 생성
# --------------------------------------------------

gb_result_pd = test_gb_pd[
    [
        "cost_center",
        "department_name",
        target,
        "simple_annualization",
        "seasonal_eac_feature"
    ]
].copy()


gb_result_pd["gradient_boosting_prediction"] = (
    gb_prediction
)


gb_result_pd["gb_error_amount"] = (
    gb_result_pd["gradient_boosting_prediction"]
    - gb_result_pd[target]
)


gb_result_pd["gb_ape"] = (
    np.abs(
        gb_result_pd["gradient_boosting_prediction"]
        - gb_result_pd[target]
    )
    / np.abs(
        gb_result_pd[target]
    )
)


gb_result_pd["seasonal_ape"] = (
    np.abs(
        gb_result_pd["seasonal_eac_feature"]
        - gb_result_pd[target]
    )
    / np.abs(
        gb_result_pd[target]
    )
)


display(
    gb_result_pd.sort_values(
        "gb_ape",
        ascending=False
    )
)

=== 2025 Forecast Comparison ===

[Simple Annualization]
MAE  : 9,566,405.76
RMSE : 20,325,575.33
MAPE : 1.0992%

[Seasonal EAC]
MAE  : 2,952,601.19
RMSE : 11,039,994.17
MAPE : 0.3213%

[Gradient Boosting]
MAE  : 25,048,625.81
RMSE : 61,049,408.89
MAPE : 2.5725%

=== Feature Importance ===
ytd_actual                    : 0.9897
cutoff_month_momentum         : 0.0051
historical_avg_ytd_share      : 0.0043
recent_3m_momentum            : 0.0009


cost_center,department_name,target_full_year_actual,simple_annualization,seasonal_eac_feature,gradient_boosting_prediction,gb_error_amount,gb_ape,seasonal_ape
410320,생산관리팀 2,1.481639304E9,1.5151123425E9,1.4833807487564223E9,1.1679765808493145E9,-3.1366272315068555E8,0.21169978570620152,0.0011753500003144363
410380,경영기획팀 3,2.98950786E8,2.97990597E8,2.9931712049809223E8,3.493182183615818E8,5.03674323615818E7,0.1684806821734926,0.0012254006854902002
410260,정보기술팀 2,1.317164518E9,1.3104314175E9,1.3468792715239115E9,1.136109354112442E9,-1.8105516388755798E8,0.13745827602650162,0.022559637097597156
410500,마케팅팀 3,7.86714864E8,6.775763535E8,7.981632917838243E8,7.085281951440687E8,-7.818666885593128E7,0.09938374426840788,0.014552194584980315
410170,연구개발팀,1.252445823E9,1.2482187075E9,1.2518174146408062E9,1.1337047085738235E9,-1.1874111442617655E8,0.09480738587299081,5.0174494389591E-4
410450,연구개발팀 3,8.67956264E8,9.268026645E8,9.337579890261233E8,9.44966127768051E8,7.700986376805103E7,0.0887255118283829,0.07581225892981548
410160,해외영업팀,1.018208935E9,1.0066617465E9,1.0195356398005837E9,9.821953502411165E8,-3.6013584758883476E7,0.035369543048533035,0.0013029789417274365
410250,재무회계팀 2,9.62334041E8,9.53412147E8,9.618768918682346E8,9.343912258834676E8,-2.7942815116532445E7,0.029036502842085834,4.750420459930146E-4
410240,경영기획팀 2,7.25948555E8,7.24513389E8,7.272159707242093E8,7.414338321149325E8,1.5485277114932537E7,0.0213310943430868,0.0017458754005086608
410510,시설관리팀 3,5.93329356E8,5.904608655E8,5.928866390926058E8,6.055797926991909E8,1.2250436699190855E7,0.020646941829709258,7.461570928814295E-4


In [0]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# --------------------------------------------------
# 1. Train / Test 데이터 준비
# --------------------------------------------------

train_rr_pd = (
    train_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "target_full_year_actual"
    )
    .orderBy(
        "fiscal_year",
        "cost_center"
    )
    .toPandas()
)


test_rr_pd = (
    test_seasonal_df
    .select(
        "fiscal_year",
        "cost_center",
        "department_name",
        "ytd_actual",
        "recent_3m_avg_actual",
        "cutoff_month_actual",
        "historical_avg_ytd_share",
        "seasonal_eac_feature",
        "target_full_year_actual"
    )
    .orderBy("cost_center")
    .toPandas()
)


# --------------------------------------------------
# 2. YTD 월평균
# --------------------------------------------------

train_rr_pd["ytd_monthly_avg"] = (
    train_rr_pd["ytd_actual"]
    / cutoff_month
)

test_rr_pd["ytd_monthly_avg"] = (
    test_rr_pd["ytd_actual"]
    / cutoff_month
)


# --------------------------------------------------
# 3. 최근 3개월 Momentum
# --------------------------------------------------

train_rr_pd["recent_3m_momentum"] = (
    train_rr_pd["recent_3m_avg_actual"]
    / train_rr_pd["ytd_monthly_avg"]
    - 1
)

test_rr_pd["recent_3m_momentum"] = (
    test_rr_pd["recent_3m_avg_actual"]
    / test_rr_pd["ytd_monthly_avg"]
    - 1
)


# --------------------------------------------------
# 4. Cutoff Month Momentum
# --------------------------------------------------

train_rr_pd["cutoff_month_momentum"] = (
    train_rr_pd["cutoff_month_actual"]
    / train_rr_pd["ytd_monthly_avg"]
    - 1
)

test_rr_pd["cutoff_month_momentum"] = (
    test_rr_pd["cutoff_month_actual"]
    / test_rr_pd["ytd_monthly_avg"]
    - 1
)


# --------------------------------------------------
# 5. Historical Pattern을 Remaining Rate 형태로 변환
# --------------------------------------------------

train_rr_pd["historical_remaining_rate"] = (
    1
    / train_rr_pd["historical_avg_ytd_share"]
    - 1
)

test_rr_pd["historical_remaining_rate"] = (
    1
    / test_rr_pd["historical_avg_ytd_share"]
    - 1
)


# --------------------------------------------------
# 6. 이번 ML이 맞힐 Target 생성
#
# Remaining Rate
# = (Full Year - YTD) / YTD
# --------------------------------------------------

train_rr_pd["target_remaining_rate"] = (
    (
        train_rr_pd["target_full_year_actual"]
        - train_rr_pd["ytd_actual"]
    )
    / train_rr_pd["ytd_actual"]
)


# YTD가 0인 경우는 Remaining Rate 계산이 불가능하므로 제외
train_rr_pd = train_rr_pd[
    train_rr_pd["ytd_actual"] != 0
].copy()

test_rr_pd = test_rr_pd[
    test_rr_pd["ytd_actual"] != 0
].copy()


# --------------------------------------------------
# 7. Feature 정의
#
# 이번에는 YTD 금액 자체를 넣지 않음
# --------------------------------------------------

rr_features = [
    "historical_remaining_rate",
    "recent_3m_momentum",
    "cutoff_month_momentum"
]


X_train_rr = train_rr_pd[rr_features]
y_train_rr = train_rr_pd["target_remaining_rate"]

X_test_rr = test_rr_pd[rr_features]


# --------------------------------------------------
# 8. Gradient Boosting 모델 생성
# --------------------------------------------------

rr_model = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=8,
    random_state=42
)


# --------------------------------------------------
# 9. Remaining Rate 학습
# --------------------------------------------------

rr_model.fit(
    X_train_rr,
    y_train_rr
)


# --------------------------------------------------
# 10. 2025 Remaining Rate 예측
# --------------------------------------------------

predicted_remaining_rate = rr_model.predict(
    X_test_rr
)


test_rr_pd["predicted_remaining_rate"] = (
    predicted_remaining_rate
)


# --------------------------------------------------
# 11. Remaining Rate → 연말 Actual로 변환
# --------------------------------------------------

test_rr_pd["gb_remaining_prediction"] = (
    test_rr_pd["ytd_actual"]
    * (
        1
        + test_rr_pd["predicted_remaining_rate"]
    )
)


# --------------------------------------------------
# 12. 평가 함수
# --------------------------------------------------

def evaluate_model(actual, prediction):

    mae = mean_absolute_error(
        actual,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )

    mask = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[mask]
                - prediction[mask]
            )
            / actual[mask]
        )
    ) * 100

    return mae, rmse, mape


actual = test_rr_pd[
    "target_full_year_actual"
]


# Seasonal EAC
seasonal_mae, seasonal_rmse, seasonal_mape = (
    evaluate_model(
        actual,
        test_rr_pd["seasonal_eac_feature"]
    )
)


# Remaining Rate Gradient Boosting
rr_mae, rr_rmse, rr_mape = (
    evaluate_model(
        actual,
        test_rr_pd["gb_remaining_prediction"]
    )
)


# --------------------------------------------------
# 13. 전체 결과 출력
# --------------------------------------------------

print("=== Seasonal EAC ===")
print(f"MAE  : {seasonal_mae:,.2f}")
print(f"RMSE : {seasonal_rmse:,.2f}")
print(f"MAPE : {seasonal_mape:.4f}%")


print("\n=== Gradient Boosting - Remaining Rate ===")
print(f"MAE  : {rr_mae:,.2f}")
print(f"RMSE : {rr_rmse:,.2f}")
print(f"MAPE : {rr_mape:.4f}%")


# --------------------------------------------------
# 14. Feature Importance
# --------------------------------------------------

print("\n=== Feature Importance ===")

for feature, importance in sorted(
    zip(
        rr_features,
        rr_model.feature_importances_
    ),
    key=lambda x: x[1],
    reverse=True
):
    print(
        f"{feature:30s}: {importance:.4f}"
    )


# --------------------------------------------------
# 15. 부서별 APE
# --------------------------------------------------

test_rr_pd["seasonal_ape"] = (
    np.abs(
        test_rr_pd["seasonal_eac_feature"]
        - actual
    )
    / np.abs(actual)
)


test_rr_pd["gb_remaining_ape"] = (
    np.abs(
        test_rr_pd["gb_remaining_prediction"]
        - actual
    )
    / np.abs(actual)
)


display(
    test_rr_pd[
        [
            "cost_center",
            "department_name",
            "ytd_actual",
            "target_full_year_actual",
            "historical_remaining_rate",
            "predicted_remaining_rate",
            "seasonal_eac_feature",
            "gb_remaining_prediction",
            "seasonal_ape",
            "gb_remaining_ape"
        ]
    ]
    .sort_values(
        "gb_remaining_ape",
        ascending=False
    )
)

=== Seasonal EAC ===
MAE  : 2,952,601.19
RMSE : 11,039,994.17
MAPE : 0.3213%

=== Gradient Boosting - Remaining Rate ===
MAE  : 6,726,835.82
RMSE : 17,169,708.32
MAPE : 0.7842%

=== Feature Importance ===
recent_3m_momentum            : 0.8302
historical_remaining_rate     : 0.1557
cutoff_month_momentum         : 0.0141


cost_center,department_name,ytd_actual,target_full_year_actual,historical_remaining_rate,predicted_remaining_rate,seasonal_eac_feature,gb_remaining_prediction,seasonal_ape,gb_remaining_ape
410500,마케팅팀 3,4.51717569E8,7.86714864E8,0.7669520659795819,0.5423891297019461,7.981632917838243E8,6.967242681209887E8,0.014552194584980315,0.11438781698042415
410450,연구개발팀 3,6.17868443E8,8.67956264E8,0.5112569667619735,0.4837647347541314,9.337579890261233E8,9.167714064408431E8,0.07581225892981548,0.05624147720978151
410340,물류팀 2,4.76837101E8,7.26068591E8,0.5212164971428375,0.5908549713827146,7.253724644909654E8,7.585786726655716E8,9.58761359000356E-4,0.04477549651444924
410260,정보기술팀 2,8.73620945E8,1.317164518E9,0.5417204443557742,0.5423891297019461,1.3468792715239115E9,1.3474634490479417E9,0.022559637097597156,0.023003148531474248
410270,인사팀 2,4.33986095E8,6.5303082E8,0.5021308123093935,0.48117987363723824,6.519038854133316E8,6.428114693524184E8,0.0017256989289852693,0.01564910925273263
410320,생산관리팀 2,1.010074895E9,1.481639304E9,0.4685849099896917,0.48117987363723824,1.4833807487564223E9,1.4961026053402467E9,0.0011753500003144363,0.009761688490039326
410310,연구개발팀 2,7.51440944E8,1.142237765E9,0.5200740096200909,0.5110461147982459,1.1422458487387862E9,1.1354619189315262E9,7.07710691584475E-6,0.005932080234165446
410120,정보기술팀,3.66000507E8,5.49216339E8,0.5017876531393672,0.5079178484681215,5.496550424553486E8,5.518986970536816E8,7.987807794418347E-4,0.004883973515000638
410520,경영기획팀 4,4.01268117E8,6.02299774E8,0.5008082322723801,0.5079178484681215,6.022264933420366E8,6.050793556454945E8,1.2166808145505815E-4,0.004614947183251742
410390,재무회계팀 3,5.50019905E8,8.25881338E8,0.5025509341184231,0.5079178484681215,8.264329220414762E8,8.293848317622405E8,6.678732356539211E-4,0.0042421272900109155


In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F


# --------------------------------------------------
# 1. Historical Pattern Gold 읽기
# --------------------------------------------------

historical_backtest_df = spark.table(
    "finance.gold.finance_historical_patterns"
)


# --------------------------------------------------
# 2. 이번에 테스트할 Forecast 시점
# --------------------------------------------------

cutoff_months = [3, 5, 8]

# 과거의 "미래"라고 가정할 연도
test_years = [2024, 2025]


summary_results = []
detail_results = []


# --------------------------------------------------
# 3. Cutoff Month별 Backtest
# --------------------------------------------------

for cutoff in cutoff_months:

    # 해당 월 시점의
    # Year × Department별 YTD / Full Year Actual
    cutoff_pd = (
        historical_backtest_df

        .filter(
            F.col("fiscal_month") == cutoff
        )

        .groupBy(
            "fiscal_year",
            "cost_center",
            "department_name"
        )

        .agg(
            F.sum(
                F.col("actual_ytd").cast("double")
            ).alias("actual_ytd"),

            F.sum(
                F.col("full_year_actual").cast("double")
            ).alias("full_year_actual")
        )

        .orderBy(
            "fiscal_year",
            "cost_center"
        )

        .toPandas()
    )


    # 각 연도에서 해당 월까지
    # 연간 Actual의 몇 %를 사용했는지 계산
    cutoff_pd["ytd_share"] = np.where(
        cutoff_pd["full_year_actual"] > 0,

        cutoff_pd["actual_ytd"]
        / cutoff_pd["full_year_actual"],

        np.nan
    )


    # --------------------------------------------------
    # 4. 2024 / 2025를 각각 Test Year로 사용
    # --------------------------------------------------

    for test_year in test_years:

        # Test Year보다 이전 연도만 Historical Pattern으로 사용
        history_pd = cutoff_pd[
            (cutoff_pd["fiscal_year"] < test_year)
            & cutoff_pd["ytd_share"].notna()
        ].copy()


        # 예측 대상 연도
        test_pd = cutoff_pd[
            cutoff_pd["fiscal_year"] == test_year
        ].copy()


        # --------------------------------------------------
        # 5. Department별 Historical YTD Share 계산
        # --------------------------------------------------

        history_summary_pd = (
            history_pd

            .groupby(
                [
                    "cost_center",
                    "department_name"
                ],
                as_index=False
            )

            .agg(
                avg_historical_ytd_share=(
                    "ytd_share",
                    "mean"
                ),

                # 과거 Share가 연도별로 얼마나 흔들렸는지
                historical_share_std=(
                    "ytd_share",
                    "std"
                ),

                history_year_count=(
                    "ytd_share",
                    "count"
                )
            )
        )


        # --------------------------------------------------
        # 6. Test Year와 Historical Pattern 결합
        # --------------------------------------------------

        backtest_pd = test_pd.merge(
            history_summary_pd,

            on=[
                "cost_center",
                "department_name"
            ],

            how="inner"
        )


        # --------------------------------------------------
        # 7. Simple Annualization
        # --------------------------------------------------

        backtest_pd["simple_annualization"] = (
            backtest_pd["actual_ytd"]
            / cutoff
            * 12
        )


        # --------------------------------------------------
        # 8. Seasonal EAC
        # --------------------------------------------------

        backtest_pd["seasonal_eac"] = (
            backtest_pd["actual_ytd"]
            / backtest_pd["avg_historical_ytd_share"]
        )


        # --------------------------------------------------
        # 9. APE 계산
        # --------------------------------------------------

        valid_actual = (
            backtest_pd["full_year_actual"] != 0
        )


        backtest_pd["simple_ape"] = np.where(
            valid_actual,

            np.abs(
                backtest_pd["simple_annualization"]
                - backtest_pd["full_year_actual"]
            )
            / np.abs(
                backtest_pd["full_year_actual"]
            ),

            np.nan
        )


        backtest_pd["seasonal_ape"] = np.where(
            valid_actual,

            np.abs(
                backtest_pd["seasonal_eac"]
                - backtest_pd["full_year_actual"]
            )
            / np.abs(
                backtest_pd["full_year_actual"]
            ),

            np.nan
        )


        # Test 정보 추가
        backtest_pd["test_year"] = test_year
        backtest_pd["cutoff_month"] = cutoff


        # --------------------------------------------------
        # 10. 전체 Summary
        # --------------------------------------------------

        summary_results.append({

            "test_year":
                test_year,

            "cutoff_month":
                cutoff,

            "history_years":
                int(
                    backtest_pd[
                        "history_year_count"
                    ].max()
                ),

            "department_count":
                len(backtest_pd),

            "simple_mape_pct":
                backtest_pd[
                    "simple_ape"
                ].mean() * 100,

            "seasonal_mape_pct":
                backtest_pd[
                    "seasonal_ape"
                ].mean() * 100,

            "seasonal_better_count":
                int(
                    (
                        backtest_pd["seasonal_ape"]
                        <
                        backtest_pd["simple_ape"]
                    ).sum()
                ),

            # Historical YTD Share의 평균 변동 정도
            "avg_historical_share_std_pct":
                backtest_pd[
                    "historical_share_std"
                ].mean() * 100
        })


        detail_results.append(
            backtest_pd
        )


# --------------------------------------------------
# 11. Summary 결과
# --------------------------------------------------

rolling_summary_pd = pd.DataFrame(
    summary_results
)

rolling_summary_pd = (
    rolling_summary_pd
    .sort_values(
        [
            "test_year",
            "cutoff_month"
        ]
    )
)


print(
    "=== Multi-Point Backtest Summary ==="
)

display(
    rolling_summary_pd
)


# --------------------------------------------------
# 12. 상세 결과 합치기
# --------------------------------------------------

rolling_detail_pd = pd.concat(
    detail_results,
    ignore_index=True
)

=== Multi-Point Backtest Summary ===


test_year,cutoff_month,history_years,department_count,simple_mape_pct,seasonal_mape_pct,seasonal_better_count,avg_historical_share_std_pct
2024,3,3,44,3.1858439383542168,1.579557012515311,41,0.26626688439945473
2024,5,3,44,1.4787917144794176,0.671598372143947,36,0.25945788301789496
2024,8,3,44,1.3838512852710727,0.6736882275270062,41,0.06379999391539232
2025,3,4,44,3.3827698814953093,1.0453413048910964,43,0.2915258102421449
2025,5,4,44,1.2390368707864343,0.19645566711809925,41,0.26503363566800764
2025,8,4,44,1.0992434753968412,0.32125850870606487,41,0.24317172853702174


In [0]:
from pyspark.sql import functions as F


# --------------------------------------------------
# 1. 분석할 10개 Scenario 부서
# --------------------------------------------------

scenario_departments = [
    "마케팅팀 3",
    "시설관리팀 3",
    "재무회계팀 4",
    "생산관리팀 2",
    "정보기술팀 2",
    "구매팀 3",
    "연구개발팀 3",
    "해외영업팀 2",
    "물류팀 2",
    "고객지원팀 3"
]


# --------------------------------------------------
# 2. 3월 / 5월 / 8월 데이터만 가져오기
#    2021~2024만 Historical 기준으로 사용
# --------------------------------------------------

scenario_pattern_df = (
    spark.table(
        "finance.gold.finance_historical_patterns"
    )

    .filter(
        (F.col("fiscal_year").between(2021, 2024))
        &
        (F.col("fiscal_month").isin(3, 5, 8))
        &
        (F.col("department_name").isin(
            scenario_departments
        ))
    )

    # Department 단위로 다시 집계
    .groupBy(
        "fiscal_year",
        "fiscal_month",
        "cost_center",
        "department_name"
    )

    .agg(
        F.sum(
            F.col("actual_ytd").cast("double")
        ).alias("department_ytd"),

        F.sum(
            F.col("full_year_actual").cast("double")
        ).alias("department_full_year")
    )

    # 각 연도의 YTD Share 계산
    .withColumn(
        "ytd_share",

        F.when(
            F.col("department_full_year") > 0,

            F.col("department_ytd")
            / F.col("department_full_year")
        )
    )
)


# --------------------------------------------------
# 3. Department × Cutoff Month별
#    Historical YTD Share 변동성 계산
# --------------------------------------------------

scenario_std_df = (
    scenario_pattern_df

    .groupBy(
        "cost_center",
        "department_name",
        "fiscal_month"
    )

    .agg(
        # 2021~2024 평균 YTD Share
        F.avg(
            "ytd_share"
        ).alias("avg_ytd_share"),

        # YTD Share 표준편차
        F.stddev_samp(
            "ytd_share"
        ).alias("std_ytd_share"),

        # 가장 낮았던 YTD Share
        F.min(
            "ytd_share"
        ).alias("min_ytd_share"),

        # 가장 높았던 YTD Share
        F.max(
            "ytd_share"
        ).alias("max_ytd_share"),

        F.count(
            "ytd_share"
        ).alias("year_count")
    )

    # 보기 편하도록 % 단위로 변환
    .withColumn(
        "avg_ytd_share_pct",
        F.col("avg_ytd_share") * 100
    )

    .withColumn(
        "std_ytd_share_pctp",
        F.col("std_ytd_share") * 100
    )

    .withColumn(
        "range_ytd_share_pctp",
        (
            F.col("max_ytd_share")
            - F.col("min_ytd_share")
        ) * 100
    )
)


# --------------------------------------------------
# 4. 결과 표시
# --------------------------------------------------

display(
    scenario_std_df

    .select(
        "cost_center",
        "department_name",
        "fiscal_month",
        "year_count",
        "avg_ytd_share_pct",
        "std_ytd_share_pctp",
        "range_ytd_share_pctp"
    )

    .orderBy(
        "department_name",
        "fiscal_month"
    )
)

cost_center,department_name,fiscal_month,year_count,avg_ytd_share_pct,std_ytd_share_pctp,range_ytd_share_pctp
410490,고객지원팀 3,3,4,24.33846010998625,0.0403827314913794,0.0928907489303854
410490,고객지원팀 3,5,4,41.13359454692777,0.05977139306523491,0.14043232064177014
410490,고객지원팀 3,8,4,66.04119622092021,0.033534385224466036,0.07277445597188015
410420,구매팀 3,3,4,24.538092028523327,0.04768407809490292,0.11574070790055235
410420,구매팀 3,5,4,41.202843966819586,0.041261183767312436,0.09929442513976205
410420,구매팀 3,8,4,66.23964052627154,0.0591115864924593,0.1280717321407554
410500,마케팅팀 3,3,4,17.578097202684575,0.20479560725153984,0.4529510005194498
410500,마케팅팀 3,5,4,35.651186131020886,0.2856806838694533,0.585603878503782
410500,마케팅팀 3,8,4,56.59463090446709,0.34484130221735854,0.8109546326326544
410340,물류팀 2,3,4,23.9082345175401,0.07970590547811958,0.1711809172315254


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np


# --------------------------------------------------
# 1. Historical Pattern 데이터 읽기
# --------------------------------------------------

hist_df = (
    spark.table(
        "finance.gold.finance_historical_patterns"
    )
    .filter(
        F.col("fiscal_year").between(2021, 2024)
    )
)


# --------------------------------------------------
# 2. 현재 Reporting Cutoff Month 확인
# --------------------------------------------------

reporting_df = spark.table(
    "finance.silver.reporting_period"
)

selector_cutoff_month = (
    reporting_df
    .orderBy(F.desc("fiscal_year"))
    .select("cutoff_month")
    .first()["cutoff_month"]
)

print(
    f"Selector Cutoff Month: "
    f"{selector_cutoff_month}"
)


# --------------------------------------------------
# 3. G/L 데이터를 Department × Month 단위로 합산
# --------------------------------------------------

dept_monthly_df = (
    hist_df

    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name",
        "fiscal_month"
    )

    .agg(
        F.sum(
            F.col("actual_amount").cast("double")
        ).alias("monthly_actual")
    )
)


# --------------------------------------------------
# 4. Department 단위 YTD / Full-Year 계산
# --------------------------------------------------

ytd_window = (
    Window

    .partitionBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )

    .orderBy("fiscal_month")

    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


full_year_window = (
    Window

    .partitionBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )
)


dept_pattern_df = (
    dept_monthly_df

    .withColumn(
        "dept_ytd_actual",

        F.sum(
            "monthly_actual"
        ).over(ytd_window)
    )

    .withColumn(
        "dept_full_year_actual",

        F.sum(
            "monthly_actual"
        ).over(full_year_window)
    )

    .withColumn(
        "ytd_share",

        F.when(
            F.col("dept_full_year_actual") > 0,

            F.col("dept_ytd_actual")
            / F.col("dept_full_year_actual")
        )
    )
)


# ==================================================
# A. SEASONALITY 안정성
# ==================================================

seasonality_df = (
    dept_pattern_df

    .filter(
        F.col("fiscal_month")
        == selector_cutoff_month
    )

    .groupBy(
        "cost_center",
        "department_name"
    )

    .agg(
        F.countDistinct(
            "fiscal_year"
        ).alias("history_year_count"),

        F.avg(
            "ytd_share"
        ).alias("avg_ytd_share"),

        F.stddev_samp(
            "ytd_share"
        ).alias("seasonality_std")
    )

    # 보기 편하도록 % 단위 생성
    .withColumn(
        "avg_ytd_share_pct",
        F.col("avg_ytd_share") * 100
    )

    .withColumn(
        "seasonality_std_pctp",
        F.col("seasonality_std") * 100
    )

    # Linear 기준과 실제 Historical Pattern의 차이
    .withColumn(
        "linear_share_gap_pctp",

        F.abs(
            F.col("avg_ytd_share")
            - F.lit(
                selector_cutoff_month / 12.0
            )
        ) * 100
    )
)


# ==================================================
# B. MONTHLY VOLATILITY
# ==================================================

year_volatility_df = (
    dept_monthly_df

    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )

    .agg(
        F.avg(
            "monthly_actual"
        ).alias("monthly_mean"),

        F.stddev_samp(
            "monthly_actual"
        ).alias("monthly_std")
    )

    .withColumn(
        "monthly_cv",

        F.when(
            F.abs(
                F.col("monthly_mean")
            ) > 0,

            F.col("monthly_std")
            / F.abs(
                F.col("monthly_mean")
            )
        )
    )
)


volatility_df = (
    year_volatility_df

    .groupBy(
        "cost_center",
        "department_name"
    )

    .agg(
        # 2021~2024의 연도별 CV 중 중앙값
        F.expr(
            "percentile_approx(monthly_cv, 0.5)"
        ).alias("monthly_volatility_cv")
    )

    .withColumn(
        "monthly_volatility_cv_pct",
        F.col("monthly_volatility_cv") * 100
    )
)


# ==================================================
# C. ANNUAL TREND
# ==================================================

annual_pd = (
    dept_monthly_df

    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )

    .agg(
        F.sum(
            "monthly_actual"
        ).alias("annual_actual")
    )

    .orderBy(
        "cost_center",
        "fiscal_year"
    )

    .toPandas()
)


trend_results = []


for (cost_center, department_name), group in (
    annual_pd.groupby(
        [
            "cost_center",
            "department_name"
        ]
    )
):

    group = group.sort_values(
        "fiscal_year"
    )

    years = group[
        "fiscal_year"
    ].astype(float).to_numpy()

    actual = group[
        "annual_actual"
    ].astype(float).to_numpy()


    # 직선 Trend 계산
    slope, intercept = np.polyfit(
        years,
        actual,
        1
    )


    prediction = (
        slope * years
        + intercept
    )


    # R² 계산
    ss_res = np.sum(
        (actual - prediction) ** 2
    )

    ss_tot = np.sum(
        (
            actual
            - np.mean(actual)
        ) ** 2
    )


    if ss_tot > 0:

        trend_r2 = (
            1
            - ss_res / ss_tot
        )

    else:

        trend_r2 = 0.0


    mean_actual = np.mean(
        np.abs(actual)
    )


    # 연간 평균 Actual 대비
    # 1년에 몇 % 수준으로 움직이는지
    if mean_actual > 0:

        trend_slope_pct_per_year = (
            slope
            / mean_actual
            * 100
        )

    else:

        trend_slope_pct_per_year = np.nan


    trend_results.append(
        {
            "cost_center":
                str(cost_center),

            "department_name":
                department_name,

            "trend_slope_pct_per_year":
                float(
                    trend_slope_pct_per_year
                ),

            "trend_r2":
                float(trend_r2)
        }
    )


trend_pd = pd.DataFrame(
    trend_results
)

trend_df = spark.createDataFrame(
    trend_pd
)


# ==================================================
# D. 세 결과 결합
# ==================================================

selector_profile_df = (
    seasonality_df

    .join(
        volatility_df,

        on=[
            "cost_center",
            "department_name"
        ],

        how="left"
    )

    .join(
        trend_df,

        on=[
            "cost_center",
            "department_name"
        ],

        how="left"
    )
)


# --------------------------------------------------
# 5. 최종 확인
# --------------------------------------------------

print(
    f"Department Count: "
    f"{selector_profile_df.count()}"
)


display(
    selector_profile_df

    .select(
        "cost_center",
        "department_name",
        "history_year_count",

        "avg_ytd_share_pct",
        "seasonality_std_pctp",
        "linear_share_gap_pctp",

        "trend_slope_pct_per_year",
        "trend_r2",

        "monthly_volatility_cv_pct"
    )

    .orderBy(
        "department_name"
    )
)

Selector Cutoff Month: 8
Department Count: 44


cost_center,department_name,history_year_count,avg_ytd_share_pct,seasonality_std_pctp,linear_share_gap_pctp,trend_slope_pct_per_year,trend_r2,monthly_volatility_cv_pct
410100,경영기획팀,4,65.9923303829739,0.020313280383174176,0.6743362836927558,5.140596704178412,0.9833754974915551,2.963753386105251
410240,경영기획팀 2,4,66.41891067367347,0.05740042419172329,0.24775599299319273,5.166397858664347,0.9817598281089475,1.6437026764201113
410380,경영기획팀 3,4,66.3712111319961,0.07511678775672055,0.2954555346705523,5.177667487984724,0.980919529995475,1.8041274252809099
410520,경영기획팀 4,4,66.63076457715692,0.09935491248859213,0.03590208950974283,5.217818193053268,0.9815715230138033,1.0678809142351788
410210,고객지원팀,4,65.94710775270272,0.049076011660852514,0.7195589139639358,5.054328329098806,0.9862850508754293,2.9330922213804698
410350,고객지원팀 2,4,66.45298278789942,0.07051868284060187,0.2136838787672457,5.162935244604283,0.9889216665807927,1.315647005363383
410490,고객지원팀 3,4,66.0411962209202,0.033534385224467396,0.6254704457464677,5.042626620495977,0.9858103089620125,2.1223308517807644
410140,구매팀,4,66.57743836970907,0.06173449767916312,0.0892282969575886,5.205345112840648,0.98604650626371,0.7457569261479028
410280,구매팀 2,4,66.43251577902443,0.02922814038967477,0.23415088764223713,5.124773358691514,0.984645175072229,1.4675403425891491
410420,구매팀 3,4,66.23964052627154,0.05911158649245929,0.4270261403951148,5.169113389562292,0.9886791060556701,2.0627160291844118


In [0]:
import pandas as pd
import numpy as np


# --------------------------------------------------
# 1. Selector Profile을 Pandas로 변환
# --------------------------------------------------

profile_pd = (
    selector_profile_df

    .select(
        "cost_center",
        "department_name",

        "seasonality_std_pctp",
        "linear_share_gap_pctp",

        "trend_slope_pct_per_year",
        "trend_r2",

        "monthly_volatility_cv_pct"
    )

    .toPandas()
)


# --------------------------------------------------
# 2. Trend의 방향이 아니라 강도를 보기 위한 값
# --------------------------------------------------

profile_pd["abs_trend_slope_pct_per_year"] = (
    profile_pd[
        "trend_slope_pct_per_year"
    ].abs()
)


# --------------------------------------------------
# 3. 분포를 확인할 지표
# --------------------------------------------------

metrics = [
    "seasonality_std_pctp",
    "linear_share_gap_pctp",
    "abs_trend_slope_pct_per_year",
    "trend_r2",
    "monthly_volatility_cv_pct"
]


# --------------------------------------------------
# 4. 각 지표의 분포 요약
# --------------------------------------------------

summary_rows = []


for metric in metrics:

    values = (
        profile_pd[metric]
        .dropna()
        .astype(float)
    )

    summary_rows.append(
        {
            "metric": metric,

            "count":
                len(values),

            "min":
                values.min(),

            "p25":
                values.quantile(0.25),

            "median":
                values.median(),

            "p75":
                values.quantile(0.75),

            "max":
                values.max()
        }
    )


profile_distribution_pd = pd.DataFrame(
    summary_rows
)


display(
    profile_distribution_pd
)

metric,count,min,p25,median,p75,max
seasonality_std_pctp,44,0.020313280383174176,0.04768044521452456,0.06176129591341399,0.08194932133426935,7.319267004787769
linear_share_gap_pctp,44,0.03590208950974283,0.23611917561243145,0.41161157971851825,0.6385186939093762,10.072035762199583
abs_trend_slope_pct_per_year,44,4.49457185899491,5.082782897890795,5.121932333142418,5.177278177307914,13.606987220895181
trend_r2,44,0.8710944189538632,0.9817127518351614,0.9836674402945417,0.9858693582874369,0.999737560576726
monthly_volatility_cv_pct,44,0.7019170666262812,1.4106446807582707,2.092523440482588,2.7219648800317664,51.81502935204112
